# Convert any webpage to Clean Markdown or Structured JSON with context.dev

This notebook walks through the three core context.dev web APIs step by step.

**Prerequisites:** Get an API key at [context.dev](https://context.dev) and set it in the cell below.

In [ ]:
!pip install -U context.dev

CONTEXT_DEV_API_KEY = "your_api_key_here"

In [ ]:
import json

from context.dev import ContextDev

client = ContextDev(api_key=CONTEXT_DEV_API_KEY)
URL = "https://stripe.com/pricing"

## 1. Single page → Clean Markdown

`web.web_scrape_md` converts a URL into GitHub Flavored Markdown, with optional main-content-only stripping.

In [ ]:
scrape = client.web.web_scrape_md(
    url=URL,
    use_main_content_only=True,
    include_links=True,
)

print(f"URL: {scrape.url}")
print(f"Credits used: {scrape.key_metadata.credits_consumed}")
print("\n--- Preview ---\n")
print(scrape.markdown[:2000])

## 2. Crawl a site → Markdown per page

`web.web_crawl_md` follows internal links and returns Markdown for each page.

In [ ]:
crawl = client.web.web_crawl_md(
    url="https://docs.stripe.com",
    max_pages=3,
    max_depth=1,
    use_main_content_only=True,
)

print(crawl.metadata)
for page in crawl.results:
    status = "ok" if page.metadata.success else "failed"
    print(f"\n[{status}] {page.metadata.title} — {page.metadata.url}")
    print(page.markdown[:500])

## 3. Extract structured JSON with a schema

`web.extract` crawls relevant pages and returns typed JSON matching your JSON Schema.

In [ ]:
pricing_schema = {
    "type": "object",
    "properties": {
        "currency": {"type": "string"},
        "plans": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "price": {"type": "string"},
                    "billing_period": {"type": "string"},
                    "features": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["name"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["plans"],
    "additionalProperties": False,
}

extracted = client.web.extract(
    url=URL,
    schema=pricing_schema,
    instructions="Prioritize the pricing page. Include currency when visible.",
    max_pages=5,
)

print("Pages analyzed:", extracted.urls_analyzed)
print(json.dumps(extracted.data, indent=2))

## Customize the schema

Change `pricing_schema` above to extract whatever you need — company info, job postings, article metadata, etc. context.dev returns JSON matching your schema.